# Adult Income — Exploratory Data Analysis

The UCI "Adult" (Census Income) dataset, drawn from the 1994 US Census. Each
row is a surveyed person; the goal of the original task is to predict whether
someone earns more than \$50K a year.

This notebook walks through a basic EDA: loading and checking the data, looking
at the target variable, handling the survey sampling weights, and building a few
visualizations. Plotly figures are interactive in the notebook (hover, zoom), and
the final view is exported to a standalone HTML file.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option("display.max_columns", None)

# Edit this path to point at your local copy of the dataset.
CSV_PATH = r"C:\Users\ryan1\Desktop\data storytelling unit\adult.csv"

## 1. Load and inspect

Missing values in this dataset are stored as the string `?` rather than as blanks,
so we pass `na_values="?"` to catch them. `skipinitialspace=True` trims the leading
spaces that appear in some of the categorical columns.

In [ ]:
df = pd.read_csv(CSV_PATH, na_values="?", skipinitialspace=True)
print("shape:", df.shape)
df.head()

In [ ]:
# How many missing values per column?
df.isna().sum()[df.isna().sum() > 0]

In [ ]:
# Drop rows with missing values for this analysis.
df = df.dropna().reset_index(drop=True)
print("rows after dropping missing:", len(df))

## 2. The target: who earns more than \$50K?

The `income` column is the label. `<=50K` means 50,000 or below; `>50K` means
above. Looking at the raw split:

In [ ]:
df["income"].value_counts(normalize=True).round(3)

About a quarter of the rows are `>50K`. That's an **imbalanced** target, which
matters for modeling later: a model that just predicts `<=50K` for everyone would
score ~76% accuracy while never identifying a single high earner. So accuracy alone
would be misleading here — precision, recall, and F1 on the `>50K` class are more
informative.

In [ ]:
income_share = (df["income"].value_counts(normalize=True) * 100).round(1)
fig = px.bar(
    x=income_share.index, y=income_share.values,
    labels={"x": "Income", "y": "Share of rows (%)"},
    title="Income distribution (raw row counts)",
    text=income_share.values,
)
fig.update_traces(textposition="outside")
fig.show()

## 3. A note on the sampling weights (`fnlwgt`)

`fnlwgt` ("final weight") is easy to misread. It is **not** a personal attribute —
it's a Census sampling weight that estimates how many people in the US population
each row represents. One row might stand in for 80,000 people, another for 200,000.

That means a raw row proportion isn't automatically a population proportion. If we
want to estimate the population share earning `>50K`, we should weight by `fnlwgt`
rather than just counting rows.

In [ ]:
raw_share = (df["income"] == ">50K").mean()

weighted_share = (
    df.loc[df["income"] == ">50K", "fnlwgt"].sum() / df["fnlwgt"].sum()
)

print(f"Unweighted (row count):   {raw_share:.3f}")
print(f"Weighted (by fnlwgt):     {weighted_share:.3f}")

The two numbers are close here but not identical — the gap is exactly what the
weight corrects for. For a population-level statement we'd cite the weighted figure;
for training a classifier, `fnlwgt` is usually dropped, since it encodes survey
design rather than anything about the person.

## 4. Deriving age groups

Binning age into a categorical makes some of the comparisons below easier to read.

In [ ]:
age_labels = ["17-24", "25-34", "35-44", "45-54", "55-64", "65+"]
df["age_group"] = pd.cut(
    df["age"], bins=[0, 25, 35, 45, 55, 65, 200],
    labels=age_labels, right=False,
)
df["age_group"].value_counts().sort_index()

## 5. Population view: each dot is a person

To get a feel for the data, we draw a sample of 1,000 people and plot them as
individual points, split by age group and colored by income. We draw the sample
**weighted by `fnlwgt`**, so the density of points reflects the population rather
than the raw rows. Hover over any point to see that person's details.

In [ ]:
sample = df.sample(n=1000, weights="fnlwgt", random_state=42)

fig = px.strip(
    sample, x="age_group", color="income",
    category_orders={"age_group": age_labels},
    hover_data=["age", "education", "occupation", "hours-per-week"],
    title="A weighted sample of 1,000 people, by age group and income",
    labels={"age_group": "Age group", "income": "Income"},
    stripmode="overlay",
)
fig.update_traces(jitter=0.35, marker={"size": 5, "opacity": 0.65})
fig.update_layout(height=500)
fig.show()

The shift is visible: the orange (`>50K`) points are sparse in the youngest
group and become a larger fraction through the middle-age brackets before thinning
out again at the top. Let's quantify that.

## 6. Income share across age groups

Here we compute the **weighted** share earning `>50K` within each age group — the
within-group rate, not the share of the whole population.

In [ ]:
def weighted_over50k(group):
    return (group.loc[group["income"] == ">50K", "fnlwgt"].sum()
            / group["fnlwgt"].sum())

by_age = (
    df.groupby("age_group", observed=True)
      .apply(weighted_over50k)
      .reindex(age_labels) * 100
).round(1)

by_age

In [ ]:
fig = px.bar(
    x=by_age.index, y=by_age.values,
    labels={"x": "Age group", "y": "% earning >50K (weighted)"},
    title="Share earning >50K by age group",
    text=by_age.values,
)
fig.update_traces(textposition="outside")
fig.update_layout(height=450)
fig.show()

## 7. Income share by education

Education is one of the stronger signals in the dataset. Same weighted calculation,
grouped by education level instead of age.

In [ ]:
by_edu = (
    df.groupby("education", observed=True)
      .apply(weighted_over50k) * 100
).sort_values().round(1)

fig = px.bar(
    x=by_edu.values, y=by_edu.index, orientation="h",
    labels={"x": "% earning >50K (weighted)", "y": "Education"},
    title="Share earning >50K by education level",
)
fig.update_layout(height=550)
fig.show()

## 8. Export an interactive figure to HTML

`fig.write_html` saves a standalone, interactive HTML file (hover and zoom still
work) that can be opened in any browser or shared. `include_plotlyjs="cdn"` keeps
the file small by loading Plotly from a CDN rather than embedding it.

In [ ]:
import webbrowser, os

out_path = os.path.join(os.path.dirname(CSV_PATH) or ".", "income_by_age.html")
fig.write_html(out_path, include_plotlyjs="cdn")
print("saved:", out_path)

webbrowser.open("file://" + os.path.realpath(out_path))